## 연결

### skills.py 수정 후 반영

- **에디터 재시작 불필요**
- `skills` 가져오기 셀(속도 설정) **다시 실행** 또는 커널 **재시작**
- `.py` 저장 후 가져오기 셀만 다시 돌리면 됨


In [2]:
# 작업 폴더 맞추기 — 상위 폴더에서 열어도 모듈 불러오기 되게
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "driver_sdk.py").exists():
    alt = ROOT / "darkroom_robot_project"
    if (alt / "driver_sdk.py").exists():
        ROOT = alt

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("작업폴더:", ROOT)
print("파이썬:", sys.executable)


작업폴더: /home/intel/darkroom_robot_project/darkroom_robot_project
파이썬: /home/intel/darkroom_robot_project/darkroom_robot_project/.venv/bin/python


In [2]:
from driver_sdk import STS3215Driver, JOINT_IDS

d = STS3215Driver("/dev/ttyACM0")
print("연결:", d.connect())

for i in JOINT_IDS:
    print(f" 서보 {i}: {'응답' if d.ping(i) else '무응답'}")


연결: True
 서보 1: 응답
 서보 2: 응답
 서보 3: 응답
 서보 4: 응답
 서보 5: 응답
 서보 6: 응답


## 토크 잠금 / 해제

- **해제** (`set_all_torque(False)`) — 손으로 움직여 티칭
- **잠금** (`set_all_torque(True)`) — 자세 고정 후 이동 명령

In [74]:
d.set_all_torque(False)  # 토크 해제 — 손으로 티칭


In [33]:
d.set_all_torque(True)  # 토크 잠금 — 이동 명령 전


In [60]:
# J6 그리퍼만 잠그고 J1~J5만 해제 (팔만 티칭할 때)
for sid in (1, 2, 3, 4, 5):
    d.set_torque(sid, False)


토크 해제 → 손으로 자세 → 아래 셀 실행 → 나온 dict를 `skills.py` 자세 상수에 반영.

In [53]:
d.get_all_positions()


{1: 2612, 2: 2696, 3: 2146, 4: 1276, 5: 989, 6: 2517}

In [ ]:
#{1: 2746, 2: 2521, 3: 2369, 4: 1240, 5: 1017, 6: 2545}

In [ ]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500) 

In [75]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500) 
d.set_all_positions({1: 2650, 2: 1992, 3: 3125, 4: 943, 5: 980, 6: 2550})

In [ ]:
#d.set_all_positions({1: 2746, 2: 2193, 3: 2734, 4: 1313, 5: 985, 6: 2550})

In [76]:
set_speeds(j1=1200, j2=300, j3=700, j4=250, j5=500, j6=500) 
d.set_all_positions({1: 2590, 2: 2409, 3: 2332, 4: 1308, 5: 1021, 6: 2510})

In [77]:
#1-1
set_speeds(j1=500, j2=200, j3=200, j4=500, j5=500, j6=500) ##
d.set_all_positions({1: 2610, 2: 2749, 3: 2144, 4: 1192, 5: 985, 6: 2520})


In [ ]:
#1-2
d.set_all_positions({1: 2610, 2: 2749, 3: 2144, 4: 1192, 5: 985, 6: 2200})

In [71]:
#2-1
d.set_all_positions({1: 2612, 2: 2696, 3: 2146, 4: 1276, 5: 989, 6: 2517})

In [72]:
#2-2
d.set_all_positions({1: 2612, 2: 2696, 3: 2146, 4: 1276, 5: 989, 6: 2300})

In [73]:
d.set_all_positions({1: 2619, 2: 2512, 3: 2467, 4: 1191, 5: 984})

In [ ]:
#set_speeds(j1=500, j2=200, j3=200, j4=700, j5=700, j6=700)
#d.set_all_positions({1: 2619, 2: 2290, 3: 2988, 4: 984, 5: 988})

In [ ]:
# set_speeds(j1=100, j2=500, j3=500, j4=500, j5=500, j6=500) 
# d.set_all_positions({1: 2625, 2: 2330, 3: 2511, 4: 1195, 5: 1021})

In [69]:
d.set_all_positions({1: 2642, 2: 2418, 3: 2497, 4: 1191, 5: 991})

In [70]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500) 
d.set_all_positions({1: 2625, 2: 1992, 3: 3125, 4: 943, 5: 980})

In [3]:
def set_speeds(j1=None, j2=None, j3=None, j4=None, j5=None, j6=None):
    """관절별 속도 설정. None이면 기본 SPEED 사용.
    예: set_speeds(j1=400, j2=300, j3=500, j4=500, j5=400, j6=500)
    """
    vals = {1: j1, 2: j2, 3: j3, 4: j4, 5: j5, 6: j6}
    for sid in JOINT_IDS:
        sp = vals[sid]
        d.set_speed(sid, SPEED if sp is None else int(sp))
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)

## 속도 설정

이동 전 토크 잠금 + 기본 속도. 각 스텝 셀에서 `set_speeds(j1=..., j2=...)` 로 관절별 조절.


In [3]:
from skills import SPEED

d.set_all_torque(True)

def set_speeds(j1=None, j2=None, j3=None, j4=None, j5=None, j6=None):
    """관절별 속도 설정. None이면 기본 SPEED 사용.
    예: set_speeds(j1=400, j2=300, j3=500, j4=500, j5=400, j6=500)
    """
    vals = {1: j1, 2: j2, 3: j3, 4: j4, 5: j5, 6: j6}
    for sid in JOINT_IDS:
        sp = vals[sid]
        d.set_speed(sid, SPEED if sp is None else int(sp))

# 기본 속도 일괄 적용
set_speeds()
print(f"토크 잠금, 기본 속도 {SPEED} — set_speeds(j1=..., j2=...) 로 스텝별 조절")


토크 잠금, 기본 속도 500 — set_speeds(j1=..., j2=...) 로 스텝별 조절


## 1차 샘플 넣기 (8스텝)

속도 기본 500. 숫자 올리면 그 관절이 먼저 도착, 내리면 늦게 도착해서 경로가 바뀜.


In [4]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본속도 — 원점으로 같이 모임
d.set_all_positions({1: 2050, 2: 745, 3: 3165, 4: 1185, 5: 1137, 6: 2200})  # 1차 0: 홈
# 팔 넓게 펼친 원점. J6=2200 닫힘 → 샘플을 이미 잡은 채 시작.


In [5]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본속도 — 대기 자세로 동시에 접힘
d.set_all_positions({1: 2003, 2: 1953, 3: 3122, 4: 910, 5: 979, 6: 2200})  # 1차 1: 준비
# J2·J3 접어 작업 대기. 그리퍼는 닫힘 유지(샘플 든 채). J1=2003·J5=979 고정.


In [6]:
set_speeds(j1=500, j2=500, j3=550, j4=360, j5=500, j6=500)  # J3 빠르게(550)·J4 느리게(360)
d.set_all_positions({1: 2003, 2: 2036, 3: 2391, 4: 1570, 5: 979, 6: 2200})  # 1차 2: 전개
# J3를 먼저 접어 팔꿈치를 당기고, J4(손목 상하)는 늦게 올려서 그리퍼가 아래로 처지며 턱에 걸리는 걸 막음.
# 손목이 먼저 올라가면 끝이 쳐져서 암실 입구에 긁힘 → J4를 느리게 해 끝이 늦게 따라오게 함.


In [7]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본속도 — J2·J3가 같이 암실 쪽으로
d.set_all_positions({1: 2003, 2: 2245, 3: 2170, 4: 1525, 5: 979, 6: 2200})  # 1차 3: 진입중
# 전개↔암실 중간. J2 더 올리고 J3 더 접음. 속도 같게 둬서 직선에 가깝게 들어감.


In [8]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본속도 — 내부 안착까지 같이 도착
d.set_all_positions({1: 2003, 2: 2454, 3: 1950, 4: 1480, 5: 979, 6: 2200})  # 1차 4: 진입
# 암실 안 샘플 놓을 위치. J6 아직 2200 닫힘. J1·J5는 그대로라 좌우·롤이 안 틀어짐.


In [16]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본(팔은 안 움직임, J6만 열림)
d.set_all_positions({1: 2003, 2: 2454, 3: 1950, 4: 1480, 5: 979, 6: 2550})  # 1차 5: 개방
# J1~J5 동일, J6만 2550. 팔 고정한 채 그리퍼만 열어 샘플을 그 자리에 내려놓음.


In [17]:
set_speeds(j1=500, j2=500, j3=1000, j4=500, j5=500, j6=500)  # 전 관절 기본속도 — 몸체·팔꿈치가 같이 후퇴
d.set_all_positions({1: 2003, 2: 2095, 3: 2590, 4: 1316, 5: 979, 6: 2550})  # 1차 6: 후퇴
# J2·J3 뒤로 빠져나옴. J4=1316(예전 1416보다 낮춤) → 손목을 조금 더 숙여 후퇴 때 끝이 암실 턱에 안 걸리게.
# 그리퍼는 2550 열림 유지(샘플은 안에 두고 나옴).


In [62]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본속도 — 대기 자세로 같이 복귀
d.set_all_positions({1: 2003, 2: 1953, 3: 3122, 4: 910, 5: 979, 6: 2550})  # 1차 7: 완료
# 준비 자세와 팔은 같고 J6만 열림. 2차 뒤집기 시작점.


## 2차 샘플 뒤집기 (13스텝)

들기 중간·헤드↑·후퇴 여러 칸을 합침. 속도↑ 그 관절 먼저 도착, 속도↓ 늦게 도착해서 경로가 휨.


In [3]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 대기 자세로 같이 모임
d.set_all_positions({1: 2003, 2: 1953, 3: 3122, 4: 910, 5: 979, 6: 2550})  # 2차 0: 대기
# 1차 완료와 동일. 그리퍼 열림, J5 중립. 여기서부터 다시 암실로 들어감.


In [4]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 입구까지 같이 펼침
d.set_all_positions({1: 2003, 2: 2036, 3: 2391, 4: 1570, 5: 979, 6: 2550})  # 2차 1: 전개
# J2·J3·J4 앞으로. 그리퍼는 연 채(다시 잡으러 감). J1·J5 고정이라 좌우·롤이 안 틀어짐.


In [5]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 직선에 가깝게 재진입
d.set_all_positions({1: 2003, 2: 2245, 3: 2170, 4: 1525, 5: 979, 6: 2550})  # 2차 2: 진입중
# 전개↔암실 중간. J2 올리고 J3 접음. 속도 같게 둬서 입구에서 끝이 아래로 처지지 않게.


In [8]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 내부까지 같이 도착
d.set_all_positions({1: 2003, 2: 2454, 3: 1950, 4: 1480, 5: 979, 6: 2550})  # 2차 3: 재진입
# 암실 안, 샘플 바로 위. 그리퍼 열림. 잡을 준비.


In [16]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 팔은 그대로, J6만 닫히면 됨
d.set_all_positions({1: 2003, 2: 2454, 3: 1950, 4: 1480, 5: 979, 6: 2200})  # 2차 4: 그립
# J1~J5 동일, J6만 2200. 위치 유지한 채 샘플만 집음.


In [26]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 어깨·팔꿈치가 같이 들림
d.set_all_positions({1: 2003, 2: 2221, 3: 2064, 4: 1480, 5: 979, 6: 2200})  # 2차 5: 들기
# 암실(J2=2454)에서 J2를 내리고 J3를 조금 펴 회전 높이로. 중간 들기 칸은 생략.
# J4=1480 유지 → 손목 각은 그대로, 샘플이 앞뒤로 까딱이지 않게.


In [33]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 실제 움직이는 건 J1뿐
d.set_all_positions({1: 2088, 2: 2221, 3: 2064, 4: 1480, 5: 979, 6: 2200})  # 2차 6: 위치
# 들기 자세 그대로, J1만 2003→2088. 베이스만 살짝 돌려 180° 롤할 좌우 여유를 만듦.


In [49]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=1000, j6=500)  # J5만 빠르게(1000) — 롤을 먼저 끝내게
d.set_all_positions({1: 2088, 2: 2221, 3: 2064, 4: 1480, 5: 3030, 6: 2200})  # 2차 7: 뒤집기
# 팔은 그대로, J5만 979→3030(180°). J5를 빠르게 해서 손목 롤이 팔보다 먼저 끝나게 함.
# 롤이 느리면 샘플이 도는 동안 팔이 같이 흔들려 덜렁임.


In [65]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 검수 쪽으로 같이
d.set_all_positions({1: 2070, 2: 2336, 3: 1972, 4: 1564, 5: 3030, 6: 2200})  # 2차 8: 검수중
# 뒤집힌 채(J5=3030) 검수 위치 중간. J4=1564로 손목을 덜 세워, 이동 중 끝이 천장·턱에 안 닿게.


In [67]:
set_speeds(j1=500, j2=500, j3=1000, j4=500, j5=500, j6=500)  # J3 빠르게(1000) — 팔꿈치를 먼저 접어 안착
d.set_all_positions({1: 2053, 2: 2469, 3: 1721, 4: 1880, 5: 3030, 6: 2200})  # 2차 9: 검수
# 검수·안착. J3를 빠르게 해서 팔꿈치가 먼저 접히고 어깨가 따라와, 위에서 수직에 가깝게 내려앉음.
# J4=1880(예전 2110보다 낮춤) → 손목을 덜 꺾어 내려놓을 때 샘플이 앞으로 쏠리지 않게.


In [97]:
set_speeds(j1=500, j2=500, j3=1000, j4=500, j5=500, j6=500)  # J3 빠르게 유지(팔은 이미 안착, J6만 열림)
d.set_all_positions({1: 2053, 2: 2469, 3: 1721, 4: 1880, 5: 3030, 6: 2550})  # 2차 10: 개방
# J1~J5 동일, J6만 2550. 검수 위치에서 그리퍼만 열어 뒤집힌 샘플을 내려놓음.


In [102]:
set_speeds(j1=500, j2=300, j3=600, j4=600, j5=1500, j6=500)  # J2 느리게(300)·J5 매우 빠르게(1500)
d.set_all_positions({1: 2046, 2: 2148, 3: 2407, 4: 1516, 5: 979, 6: 2550})  # 2차 11: 후퇴중
# 헤드↑+후퇴를 한 칸으로 합침. J5=1500으로 롤을 먼저 중립(979)까지 되돌림 → 손목이 다 돈 뒤에 팔이 움직임(놓아둔 샘플을 안 건드림).
# J2=300으로 어깨는 늦게 빠져나와, 끝이 암실에 남은 채 몸만 확 빠지는 걸 막음. J3·J4는 600으로 조금 빨리 접히며 따라옴.


In [103]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 대기로 같이 복귀
d.set_all_positions({1: 2003, 2: 1953, 3: 3122, 4: 910, 5: 979, 6: 2550})  # 2차 12: 완료
# 대기 복귀, 그리퍼 열림. 3차 꺼내기 시작점.


## 3차 샘플 꺼내기 (7스텝)

들기 칸 생략, 그립 후 바로 퇴출. 마지막에 J5 롤을 먼저 돌려 샘플을 들고 나옴.


In [141]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본
d.set_all_positions({1: 2003, 2: 1953, 3: 3122, 4: 910, 5: 979, 6: 2550})  # 3차 0: 대기
# 2차 완료와 동일. 그리퍼 열림.


In [133]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 입구까지 같이
d.set_all_positions({1: 2003, 2: 2036, 3: 2391, 4: 1570, 5: 979, 6: 2550})  # 3차 1: 전개
# 1·2차와 같은 진입 경로. 그리퍼 연 채 암실 입구.


In [134]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본 — 휘지 않게 같이 진입
d.set_all_positions({1: 2003, 2: 2245, 3: 2170, 4: 1525, 5: 979, 6: 2550})  # 3차 2: 진입중
# 전개↔암실 중간. 속도 같게 둬서 끝이 아래로 처지지 않게.


In [137]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 전 관절 기본
d.set_all_positions({1: 2003, 2: 2454, 3: 1950, 4: 1480, 5: 979, 6: 2550})  # 3차 3: 진입
# 암실 내부, 그리퍼 열림. 샘플 잡을 위치.


In [138]:
set_speeds(j1=500, j2=500, j3=500, j4=500, j5=500, j6=500)  # 팔 고정, J6만 닫힘
d.set_all_positions({1: 2003, 2: 2454, 3: 1950, 4: 1480, 5: 979, 6: 2200})  # 3차 4: 그립
# 같은 자리에서 J6만 닫아 샘플 집음. 들기 중간은 생략하고 바로 퇴출.


In [139]:
set_speeds(j1=500, j2=500, j3=1000, j4=500, j5=500, j6=500)  # J3 빠르게(1000) — 팔꿈치를 먼저 접어 빼냄
d.set_all_positions({1: 2003, 2: 2087, 3: 2593, 4: 1195, 5: 979, 6: 2200})  # 3차 5: 퇴출중
# 들기+후퇴를 한 칸으로. J3를 빠르게 해서 팔꿈치가 먼저 접히고, 어깨는 기본 속도로 따라옴
# → 끝이 암실에 남은 채 몸만 빠지지 않음. J4=1195로 손목을 낮춰 들고 나올 때 턱에 안 걸리게. 그리퍼 닫힘.


In [140]:
set_speeds(j1=500, j2=500, j3=1000, j4=500, j5=1500, j6=500)  # J3 빠르게(1000)·J5 매우 빠르게(1500)
d.set_all_positions({1: 2003, 2: 1953, 3: 3122, 4: 910, 5: 3030, 6: 2200})  # 3차 6: 퇴출
# 대기 자세로 복귀하되 J6 닫힘(샘플 든 채). J5=3030이라 들고 나오면서 손목 롤 180°를 같이 끝냄.
# J5를 1500으로 먼저 돌려 샘플이 도는 동안 팔이 덜 흔들리게 하고, J3=1000으로 팔꿈치를 빨리 접어 벽에 안 닿게 함.
